In [1]:
pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00


In [2]:
import torch, random, numpy as np
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

In [3]:
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [4]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model and preprocess function

In [5]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

/usr/local/lib/python3.13/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


### Load the dataset and prepare the classnames

In [7]:
from datasets import load_dataset

ds = load_dataset("axiong/imagenet-r")

In [8]:
unique_pairs = sorted(set(zip(ds['test']["wnid"], ds['test']["class_name"])))
r_wnids = [pair[0] for pair in unique_pairs]
r_class_names = [pair[1].replace('_', ' ') for pair in unique_pairs]

In [9]:
len(r_class_names)

200

In [10]:
r_class_names[:5]

['goldfish', 'great white shark', 'hammerhead', 'stingray', 'hen']

In [11]:
wnid_to_r_index = {wnid: i for i, wnid in enumerate(r_wnids)}

for wnid, idx in list(wnid_to_r_index.items())[:5]:
  print(f"wnid: {wnid}, idx: {idx}")

wnid: n01443537, idx: 0
wnid: n01484850, idx: 1
wnid: n01494475, idx: 2
wnid: n01498041, idx: 3
wnid: n01514859, idx: 4


### Prepare the few shot and test data

In [12]:
from collections import defaultdict

classes_to_indices = defaultdict(list)
all_indices = []

for idx in range(len(ds['test'])):
  label = ds['test'][idx]['class_name'].replace('_', ' ')
  classes_to_indices[label].append(idx)
  all_indices.append(idx)

In [13]:
print(len(classes_to_indices))
print(len(all_indices))

200
30000


In [14]:
few_shot_indices = []

for cls, indices in classes_to_indices.items():
  sampled = random.sample(indices, 16)
  few_shot_indices.extend(sampled)

In [15]:
len(few_shot_indices) == (200 * 16)

True

In [16]:
class HFImageDataset(Dataset):
    def __init__(self, hf_dataset, preprocess, wnid_to_index):
        self.hf_dataset = hf_dataset
        self.preprocess = preprocess
        self.wnid_to_index = wnid_to_index

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        example = self.hf_dataset[idx]
        image = self.preprocess(example["image"].convert("RGB"))
        label = self.wnid_to_index[example["wnid"]]
        return image, label

In [17]:
full_wrapped = HFImageDataset(ds['test'], preprocess, wnid_to_r_index)

In [18]:
from torch.utils.data import Subset

raw_few_shot_ds = Subset(full_wrapped, few_shot_indices)
print(len(raw_few_shot_ds))

3200


In [19]:
eval_indices = list(set(all_indices) - set(few_shot_indices))
print(len(eval_indices))

26800


In [20]:
raw_eval_ds = Subset(full_wrapped, eval_indices)
print(len(raw_eval_ds))

26800


In [21]:
raw_few_shot_loader = DataLoader(raw_few_shot_ds, batch_size = 32, shuffle = True)
raw_eval_loader = DataLoader(raw_eval_ds, batch_size = 32)

### Build the cache model for Tip-Adapter

In [21]:
mkdir features

In [31]:
from clip_zeroshot import build_and_cache_image_features, build_and_cache_text_features

In [23]:
few_shot_image_cache = build_and_cache_image_features(model, device, raw_few_shot_loader, './features', 'few_shot_image_features')

  0%|          | 0/100 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/few_shot_image_features.pt


In [ ]:
# few_shot_image_features = few_shot_image_cache['image_features']
# few_shot_image_labels = few_shot_image_cache['labels']

In [23]:
# CAUTION: Only use if to load cached features

from clip_zeroshot import load_cached_image_features

loaded_few_shot_image_cache = load_cached_image_features('/content/features/few_shot_image_features.pt')

few_shot_image_features = loaded_few_shot_image_cache['image_features']
few_shot_image_labels = loaded_few_shot_image_cache['labels']

In [24]:
print(few_shot_image_features.shape)
print(few_shot_image_labels.shape)

torch.Size([3200, 512])
torch.Size([3200])


In [25]:
few_shot_image_labels[:10]

tensor([150, 167, 105, 156, 151, 119, 195,   1,  24, 170])

In [26]:
import torch.nn.functional as F

one_hot = F.one_hot(few_shot_image_labels, num_classes=len(r_class_names))

In [27]:
print(one_hot.shape)

torch.Size([3200, 200])


In [28]:
cache_keys = few_shot_image_features
cache_values = one_hot.float()

### Build the zero shot classifier

In [29]:
from imagenet_classes import IMAGENET_TEMPLATES

In [32]:
text_features = build_and_cache_text_features(model = model, device= device, tokenizer = tokenizer, classnames = r_class_names, templates = IMAGENET_TEMPLATES, cache_dir='./features', file_name='text-features')

  0%|          | 0/200 [00:00<?, ?it/s]

Text features has been saved at ./features/text-features.pt


### Build the test features

In [32]:
# eval_cache = build_and_cache_image_features(model, device, raw_eval_loader, './features', 'r_eval_features')
# eval_features = eval_cache['image_features']
# eval_labels = eval_cache['labels']

  0%|          | 0/838 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/r_eval_features.pt


In [33]:
# CAUTION: Only use it to load cached features

from clip_zeroshot import load_cached_image_features

load_eval_cached = load_cached_image_features('/content/features/r_eval_features.pt')
eval_features = load_eval_cached['image_features']
eval_labels = load_eval_cached['labels']

### Build the coop's text features

In [34]:
from coop import PromptLearner, TextEncoderWrapper

In [35]:
text_encoder = TextEncoderWrapper(model)
coop_prompt_learner = PromptLearner(clip_model=model, device=device, n_ctx=4, tokenizer=tokenizer, ctx_dim=512, class_names=r_class_names).to(device)
coop_prompt_learner.ctx.data.copy_(torch.load('features/coop_ctx.pt'))

with torch.no_grad():
    prompts, tok = coop_prompt_learner()
    coop_text_features = text_encoder(prompts, tok)
    coop_text_features = coop_text_features / coop_text_features.norm(dim=-1, keepdim=True)

### Run the harness

In [36]:
from harness import run_comparison, zero_shot_logits, tip_adapter_logits, coop_logits

In [37]:
def accuracy(logits, labels):
    preds = logits.argmax(dim=-1)
    return 100 * (preds == labels).float().mean().item()

metrics = {"accuracy": accuracy}

In [38]:
shared = {
    "test_features": eval_features.to(device),
    "labels": eval_labels.to(device),
    "text_features": text_features.to(device),
    "cache_keys": cache_keys.to(device),
    "cache_values": cache_values.to(device),
    "logit_scale": model.logit_scale.exp(),
    "coop_text_features": coop_text_features.to(device)
}

In [39]:
methods = {
    "zero_shot":   {"fn": zero_shot_logits,   "params": {}},
    "tip_adapter": {"fn": tip_adapter_logits, "params": {"alpha": 1.5, "beta": 5.0}},
    "coop":        {"fn": coop_logits,        "params": {}}
}

In [40]:
results = run_comparison(shared, methods, metrics)
print(results)

{'zero_shot': {'accuracy': 73.54104518890381}, 'tip_adapter': {'accuracy': 74.86193776130676}, 'coop': {'accuracy': 75.07089376449585}}


### Get the context from coop

In [38]:
# from coop import PromptLearner, TextEncoderWrapper

In [39]:
# prompt_learner = PromptLearner(clip_model=model, device=device, n_ctx=4, tokenizer=tokenizer, ctx_dim=512, class_names=r_class_names).to(device)

In [40]:
# text_encoder = TextEncoderWrapper(model)

In [41]:
# from torch.utils.data import TensorDataset

# img_features_dataset = TensorDataset(few_shot_image_features, few_shot_image_labels)
# train_loader = DataLoader(img_features_dataset, batch_size=32, shuffle=True)

In [42]:
# for param in model.parameters():
#   param.requires_grad_(False)

In [43]:
# from tqdm.notebook import tqdm
# import torch.nn.functional as F

# epochs = 10
# optimizer = torch.optim.Adam(prompt_learner.parameters(), lr=0.002)
# num_ctx = 4
# ctx_dim = 312
# logit_scale = model.logit_scale.exp()

# for epoch in range(epochs+1):
#   total_loss = 0
#   for img_feat, labels in tqdm(train_loader):
#     img_feat = img_feat.to(device)
#     labels = labels.to(device)

#     prompts, tok_prompts = prompt_learner()
#     text_features = text_encoder(prompts, tok_prompts)
#     text_features = text_features / text_features.norm(dim=-1,keepdim=True)

#     logits = logit_scale * img_feat @ text_features.t()
#     loss = F.cross_entropy(logits, labels)
#     total_loss += loss.item()

#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

#   epoch_loss = total_loss / len(train_loader)
#   print(f"epoch : {epoch + 1}, loss : {epoch_loss: .4f}")

  0%|          | 0/100 [00:00<?, ?it/s]

epoch : 1, loss :  1.0773


  0%|          | 0/100 [00:00<?, ?it/s]

epoch : 2, loss :  1.0040


  0%|          | 0/100 [00:00<?, ?it/s]

...

epoch : 9, loss :  0.8363


  0%|          | 0/100 [00:00<?, ?it/s]

epoch : 10, loss :  0.8166


  0%|          | 0/100 [00:00<?, ?it/s]

epoch : 11, loss :  0.8041


In [44]:
# torch.save(prompt_learner.ctx.data, 'features/coop_ctx.pt')

In [45]:
# eval_feature_dataset = TensorDataset(eval_features, eval_labels)
# test_loader = DataLoader(eval_feature_dataset, batch_size=32, shuffle=False)

In [47]:
# prompt_learner.eval()

# with torch.no_grad():
#     prompts, tokenized = prompt_learner()
#     text_features = text_encoder(prompts, tokenized)
#     text_features = text_features / text_features.norm(dim=-1, keepdim=True)

#     correct = 0
#     total = 0
#     for image_features, labels in test_loader:
#         image_features = image_features.to(device)
#         labels = labels.to(device)

#         logits = image_features @ text_features.t()
#         preds = logits.argmax(dim=-1)

#         correct += (preds == labels).sum().item()
#         total += labels.size(0)

# accuracy = 100 * correct / total
# print(f"Test accuracy: {accuracy:.2f}")

Test accuracy: 75.07
